#NB_hospital_daily_summary

In [0]:
import time

_run_start = time.time()
target_table = "adb_healthcare.gold.hospital_daily_summary"
print("=" * 70)
print(f"[START] NB_hospital_daily_summary building '{target_table}'")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend (STTM silver to gold)'
AS
WITH lab_results_dedup AS (
  -- silver.lab_results contains every test once per load batch: keep the latest version of each test
  SELECT *
  FROM adb_healthcare.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY landing_timestamp DESC, update_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                               AS summary_date,
    count(*)                                AS total_tests,
    count(DISTINCT patient_id)              AS unique_patients_tested,
    count(DISTINCT doctor_id)               AS active_doctor_count,
    count_if(result_status = 'Abnormal')    AS abnormal_test_count,
    count_if(result_status = 'Critical')    AS critical_test_count
  FROM lab_results_dedup
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    lag(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                                     AS hospital_id,
  h.hospital_name                                                                AS hospital_name,
  w.summary_date                                                                 AS summary_date,
  c.day_name                                                                     AS day_name,     -- e.g. Monday
  c.is_weekend                                                                   AS is_weekend,
  CAST(w.total_tests AS INT)                                                     AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                          AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                             AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                             AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                             AS critical_test_count,
  CAST(round(w.abnormal_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2))    AS abnormal_rate_pct,
  CAST(round(w.critical_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2))    AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                            AS prev_day_total_tests,  -- NULL for a hospital's first day
  CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
       ELSE CAST(round((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100, 2) AS DECIMAL(6,2))
  END                                                                            AS test_volume_change_pct -- day-over-day % change
FROM with_lag w
LEFT JOIN adb_healthcare.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_healthcare.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:
records_count = spark.table(target_table).count()
print(f"[EXIT] Returning records_count = {records_count} to caller")
print(f"[END] NB_hospital_daily_summary finished for table '{target_table}' in {time.time() - _run_start:.1f}s")
print("=" * 70)
dbutils.notebook.exit(str(records_count))